# Heart Disease Prediction with PCA

Dataset credit: https://www.kaggle.com/fedesoriano/heart-failure-prediction

Goal: clean the dataset, encode categorical features, scale the data, compare SVM, Logistic Regression, and Random Forest models, then repeat the comparison after PCA.


## 1. Load the dataset


In [1]:
import pandas as pd
import numpy as np

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC


In [2]:
df = pd.read_csv("heart.csv")
df.head()


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [3]:
print("Shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
df.info()


Shape: (918, 12)
Missing values: 0
<class 'pandas.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    str    
 2   ChestPainType   918 non-null    str    
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    str    
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    str    
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    str    
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), str(5)
memory usage: 97.6 KB


In [4]:
df.describe()


,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


## 2. Remove outliers using Z-score

The target column `HeartDisease` is not used for outlier detection. A row is removed if any numeric feature has a Z-score greater than 3 or less than -3.


In [5]:
numeric_feature_cols = df.select_dtypes(include=np.number).columns.drop("HeartDisease")

z_scores = (df[numeric_feature_cols] - df[numeric_feature_cols].mean()) / df[numeric_feature_cols].std()
outlier_mask = (z_scores.abs() > 3).any(axis=1)

outliers = df.loc[outlier_mask].copy()
df_clean = df.loc[~outlier_mask].copy()

print("Original shape:", df.shape)
print("Outliers removed:", outlier_mask.sum())
print("Cleaned shape:", df_clean.shape)
outliers


Original shape: (918, 12)
Outliers removed: 19
Cleaned shape: (899, 12)


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
76,32,M,ASY,118,529,0,Normal,130,N,0.0,Flat,1
109,39,M,ATA,190,241,0,Normal,106,N,0.0,Up,0
149,54,M,ASY,130,603,1,Normal,125,Y,1.0,Flat,1
166,50,M,ASY,140,231,0,ST,140,Y,5.0,Flat,1
241,54,M,ASY,200,198,0,Normal,142,Y,2.0,Flat,1
324,46,M,ASY,100,0,1,ST,133,N,-2.6,Flat,1
365,64,F,ASY,200,0,0,Normal,140,Y,1.0,Flat,1
390,51,M,ASY,140,0,0,Normal,60,N,0.0,Flat,1
399,61,M,NAP,200,0,1,ST,70,N,0.0,Flat,1
449,55,M,NAP,0,0,0,Normal,155,N,1.5,Flat,1


## 3. Encode text columns

Binary text columns are label encoded. Multi-category text columns are one-hot encoded.


In [6]:
df_encoded = df_clean.copy()

label_encoded_cols = ["Sex", "ExerciseAngina"]
label_encoders = {}

for col in label_encoded_cols:
    encoder = LabelEncoder()
    df_encoded[col] = encoder.fit_transform(df_encoded[col])
    label_encoders[col] = encoder

one_hot_cols = ["ChestPainType", "RestingECG", "ST_Slope"]
df_encoded = pd.get_dummies(df_encoded, columns=one_hot_cols, drop_first=True, dtype=int)

print("Encoded shape:", df_encoded.shape)
print("Object columns left:", df_encoded.select_dtypes(include="object").columns.tolist())
df_encoded.head()


Encoded shape: (899, 16)
Object columns left: []


,Age,Sex,RestingBP,Cholesterol,FastingBS,MaxHR,ExerciseAngina,Oldpeak,HeartDisease,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ST_Slope_Flat,ST_Slope_Up
0,40,1,140,289,0,172,0,0.0,0,1,0,0,1,0,0,1
1,49,0,160,180,0,156,0,1.0,1,0,1,0,1,0,1,0
2,37,1,130,283,0,98,0,0.0,0,1,0,0,0,1,0,1
3,48,0,138,214,0,108,1,1.5,1,0,0,0,1,0,1,0
4,54,1,150,195,0,122,0,0.0,0,0,1,0,1,0,0,1


## 4. Split and scale the data

The train/test split happens before scaling so the scaler is fitted only on training data.


In [7]:
X = df_encoded.drop("HeartDisease", axis=1)
y = df_encoded["HeartDisease"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train:", X_train_scaled.shape)
print("X_test:", X_test_scaled.shape)


X_train: (719, 15)
X_test: (180, 15)


## 5. Train classification models without PCA


In [8]:
models = {
    "SVM": SVC(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

baseline_scores = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    baseline_scores[name] = accuracy_score(y_test, y_pred)

baseline_results = pd.DataFrame({
    "Model": baseline_scores.keys(),
    "Accuracy without PCA": baseline_scores.values(),
}).sort_values("Accuracy without PCA", ascending=False)

baseline_results


,Model,Accuracy without PCA
0,SVM,0.894444
1,Logistic Regression,0.872222
2,Random Forest,0.872222


## 6. Apply PCA and retrain models

PCA is fitted only on the scaled training data, then the same transformation is applied to the test data.


In [9]:
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Original number of features:", X_train_scaled.shape[1])
print("PCA components retained:", pca.n_components_)
print("Explained variance retained:", round(pca.explained_variance_ratio_.sum(), 4))


Original number of features: 15
PCA components retained: 13
Explained variance retained: 0.972


In [10]:
pca_scores = {}

for name, model in models.items():
    model.fit(X_train_pca, y_train)
    y_pred = model.predict(X_test_pca)
    pca_scores[name] = accuracy_score(y_test, y_pred)

comparison = pd.DataFrame({
    "Model": baseline_scores.keys(),
    "Accuracy without PCA": baseline_scores.values(),
    "Accuracy with PCA": [pca_scores[name] for name in baseline_scores.keys()],
})
comparison["Accuracy Difference"] = comparison["Accuracy with PCA"] - comparison["Accuracy without PCA"]
comparison["PCA Components"] = pca.n_components_
comparison = comparison.sort_values("Accuracy without PCA", ascending=False).reset_index(drop=True)

comparison


,Model,Accuracy without PCA,Accuracy with PCA,Accuracy Difference,PCA Components
0,SVM,0.894444,0.888889,-0.005556,13
1,Logistic Regression,0.872222,0.872222,0.000000,13
2,Random Forest,0.872222,0.894444,0.022222,13


In [11]:
best_without_pca = comparison.loc[comparison["Accuracy without PCA"].idxmax()]
best_with_pca = comparison.loc[comparison["Accuracy with PCA"].idxmax()]

print(
    f"Best model without PCA: {best_without_pca['Model']} "
    f"({best_without_pca['Accuracy without PCA']:.4f})"
)
print(
    f"Best model with PCA: {best_with_pca['Model']} "
    f"({best_with_pca['Accuracy with PCA']:.4f})"
)

if best_with_pca["Accuracy with PCA"] < best_without_pca["Accuracy without PCA"]:
    print("PCA reduced dimensionality but slightly reduced accuracy in this run.")
elif best_with_pca["Accuracy with PCA"] > best_without_pca["Accuracy without PCA"]:
    print("PCA reduced dimensionality and improved the best accuracy in this run.")
else:
    print("PCA reduced dimensionality while keeping the best accuracy unchanged in this run.")


Best model without PCA: SVM (0.8944)
Best model with PCA: Random Forest (0.8944)
PCA reduced dimensionality while keeping the best accuracy unchanged in this run.


## Conclusion

The PCA version uses fewer dimensions, which can make computation lighter. The tradeoff is that accuracy may decrease, stay similar, or sometimes improve depending on the model and the data split. The final comparison table above shows the impact for this dataset.
